# 02 · Build the training dataset (Kaggle, CPU session)

Converts every labelled frame once (`python -m visdet.prepare`):
- halves frames until they are ≤ 1920 px wide (the test resolution groups),
- fills ignore regions with gray and drops boxes > 50 % inside them,
- writes YOLO labels, the box/image tables (test-scale pixels) and the clip → fold assignment.

**Kaggle:** add the competition data, choose *CPU* (no GPU quota needed), *Save & Run All*. The output
(`prepared/` ~9 GB and `prepared_nomask/` ~1 GB, ~1.5 h) is then attached to the GPU notebooks via
*Add Input → Notebook Output*.

In [ ]:
# Setup: code + extra packages. On Kaggle the repo comes from an attached dataset or GitHub.
import os, sys, shutil, subprocess
from pathlib import Path

GITHUB_REPO = "<github-user>/<repo>"   # your repository; if private, add a GITHUB_TOKEN Kaggle secret
ON_KAGGLE = Path("/kaggle/working").exists()
if ON_KAGGLE:
    REPO = Path("/kaggle/working/repo")
    if not REPO.exists():
        code = [p.parent for p in Path("/kaggle/input").glob("*/pyproject.toml")]
        if code:                                   # repo uploaded as a Kaggle dataset
            shutil.copytree(code[0], REPO)
        else:                                      # clone from GitHub
            try:
                from kaggle_secrets import UserSecretsClient
                auth = UserSecretsClient().get_secret("GITHUB_TOKEN") + "@"
            except Exception:
                auth = ""
            subprocess.run(["git", "clone", "-q", f"https://{auth}github.com/{GITHUB_REPO}.git", str(REPO)], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "-r", str(REPO / "requirements.txt")], check=True)
    sys.path.insert(0, str(REPO / "src"))
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO)
print("repo:", REPO)

In [ ]:
# Convert all frames (step 1); experiments later choose every 10th / 5th / 3rd frame from these lists
OUT_DIR = Path("/kaggle/working/prepared") if ON_KAGGLE else REPO / "prepared"
!python -m visdet.prepare --out {OUT_DIR} --step 1 --workers {os.cpu_count()} 2>&1 | tail -n 1
# Variant for the ignore-region experiment (notebook 07): every 10th frame, regions left visible, labels still dropped
!python -m visdet.prepare --out {OUT_DIR}_nomask --step 10 --no-mask --workers {os.cpu_count()} 2>&1 | tail -n 1

In [ ]:
# Sanity checks: frames written, label files consistent with the box table, disk usage
from visdet.data import read_prepared
images, boxes, folds = read_prepared(OUT_DIR)
n_img = sum(1 for _ in (OUT_DIR / "images").rglob("*.jpg"))
n_lbl = sum(len(p.read_text().splitlines()) for p in (OUT_DIR / "labels").rglob("*.txt"))
print(f"frames: {n_img} written / {images.prepared.sum()} expected; label lines: {n_lbl} / {boxes.keep.sum()} expected")
print("clips per fold:", folds.groupby("fold").size().to_dict())
!du -sh {OUT_DIR}